# Roadmaps MVP End-to-End

This notebook walks through the current Python MVP pipeline for certification-program planning:

1. validate raw extracted course JSON files
2. inspect the skill taxonomy and alias normalization layer
3. transform raw courses into canonical course objects and quality snapshots
4. load the semester-based Data Science track configuration
5. match courses into the chosen track
6. inspect semester-by-semester candidate ranking, blocked reasons, and final roadmap
7. compare the successful baseline against an infeasible weaker baseline
8. run the public CLI commands and inspect their outputs

The notebook is designed to run from any working directory inside the repository tree.

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'roadmaps_mvp').exists():
            return candidate
    raise RuntimeError('Could not detect the roadmaps repo root.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from roadmaps_mvp.io import iter_raw_course_paths, load_json
from roadmaps_mvp.models import ProgramSkillLevel
from roadmaps_mvp.normalize import SkillResolver, load_taxonomy
from roadmaps_mvp.planner import build_catalog, build_roadmap, load_program, rank_track_courses
from roadmaps_mvp.validate import validate_raw_course_dir

PROGRAM_PATH = REPO_ROOT / 'programs' / 'data_science_mvp.json'
TAXONOMY_PATH = REPO_ROOT / 'taxonomies' / 'skills.json'
REPORT_PATH = REPO_ROOT / 'reports' / 'roadmap_result.json'
NOTEBOOK_PATH = REPO_ROOT / 'output' / 'jupyter-notebook' / 'roadmaps-mvp-end-to-end.ipynb'


def show(title: str, payload, limit: int | None = None):
    print(f'\n{title}')
    print('-' * len(title))
    if hasattr(payload, 'model_dump'):
        payload = payload.model_dump(mode='json')
    if isinstance(payload, list) and limit is not None:
        payload = payload[:limit]
    print(json.dumps(payload, indent=2, ensure_ascii=False))


print('Current working directory:', Path.cwd())
print('Detected repo root:', REPO_ROOT)
print('Program path:', PROGRAM_PATH)
print('Notebook path:', NOTEBOOK_PATH)

Current working directory: /Users/pelmeshek1706/Desktop/projects/roadmaps/output/jupyter-notebook
Detected repo root: /Users/pelmeshek1706/Desktop/projects/roadmaps
Program path: /Users/pelmeshek1706/Desktop/projects/roadmaps/programs/data_science_mvp.json
Notebook path: /Users/pelmeshek1706/Desktop/projects/roadmaps/output/jupyter-notebook/roadmaps-mvp-end-to-end.ipynb


## 1. Validate raw course JSON files

The repository keeps the original discipline descriptions as a stable raw layer. We validate those files first and do not mutate them into program objects.

In [3]:
raw_paths = list(iter_raw_course_paths(REPO_ROOT))
raw_courses, validation_issues = validate_raw_course_dir(REPO_ROOT)

show(
    'Raw course files',
    [
        {
            'file': path.name,
            'course_id': path.stem,
        }
        for path in raw_paths
    ],
)
show(
    'Validation issues',
    [issue.model_dump(mode='json') for issue in validation_issues],
)
print(f'Validated courses: {len(raw_courses)}')


Raw course files
----------------
[
  {
    "file": "analysis_and_processing_of_time_series.json",
    "course_id": "analysis_and_processing_of_time_series"
  },
  {
    "file": "computer_vision_technologies.json",
    "course_id": "computer_vision_technologies"
  },
  {
    "file": "data_science_technologies.json",
    "course_id": "data_science_technologies"
  },
  {
    "file": "fundamentals_of_data_science.json",
    "course_id": "fundamentals_of_data_science"
  },
  {
    "file": "natural_language_analysis_and_processing_nlp.json",
    "course_id": "natural_language_analysis_and_processing_nlp"
  }
]

Validation issues
-----------------
[]
Validated courses: 5


## 2. Inspect one raw course object

This shows the extraction-layer shape before canonical normalization.

In [4]:
sample_raw_path = next(path for path in raw_paths if path.stem == 'data_science_technologies')
sample_raw = load_json(sample_raw_path)
show(
    'Raw course preview',
    {
        'course_name': sample_raw['course_name'],
        'discipline_tags': sample_raw['discipline_tags'],
        'input_skills_normalized_count': len(sample_raw['input_skills_normalized']),
        'output_skills_normalized_count': len(sample_raw['output_skills_normalized']),
        'curricular_relations_count': len(sample_raw['curricular_relations']),
        'confidence': sample_raw['confidence'],
    },
)
show('First 2 raw input skills', sample_raw['input_skills_normalized'], limit=2)
show('First 2 raw output skills', sample_raw['output_skills_normalized'], limit=2)


Raw course preview
------------------
{
  "course_name": "Data Science Technologies",
  "discipline_tags": [
    "data science",
    "statistical learning",
    "machine learning",
    "artificial intelligence",
    "decision support systems",
    "data mining",
    "time series analytics",
    "geospatial analytics"
  ],
  "input_skills_normalized_count": 7,
  "output_skills_normalized_count": 18,
  "curricular_relations_count": 11,
  "confidence": "medium"
}

First 2 raw input skills
------------------------
[
  {
    "skill_id": "python_basics",
    "skill_label": "Python basics",
    "min_level_required": 3,
    "importance": "required",
    "source": "explicit",
    "confidence": "high",
    "raw_mentions": [
      "Python syntax",
      "types and data structures",
      "branching operators",
      "functional programming",
      "OOP programming",
      "working with IDE",
      "environment creation"
    ],
    "evidence": [
      "The syllabus explicitly requires basic Pytho

## 3. Skill taxonomy and alias normalization

The taxonomy layer is the source of truth for canonical skill IDs. Raw aliases are resolved into canonical IDs before any cross-course planning happens.

In [5]:
taxonomy = load_taxonomy(TAXONOMY_PATH)
resolver = SkillResolver(taxonomy)

show(
    'Taxonomy summary',
    {
        'taxonomy_id': taxonomy.taxonomy_id,
        'version': taxonomy.version,
        'skills_count': len(taxonomy.skills),
    },
)
show(
    'Alias resolution examples',
    [
        {'query': 'basic_programming', 'resolved_to': resolver.resolve('basic_programming').canonical_skill_id},
        {'query': 'programming_basics', 'resolved_to': resolver.resolve('programming_basics').canonical_skill_id},
        {'query': 'linear_algebra_basics', 'resolved_to': resolver.resolve('linear_algebra_basics').canonical_skill_id},
        {'query': 'data_structures_and_algorithms', 'resolved_to': resolver.resolve('data_structures_and_algorithms').canonical_skill_id},
    ],
)
show('First 8 taxonomy entries', [entry.model_dump(mode='json') for entry in taxonomy.skills], limit=8)


Taxonomy summary
----------------
{
  "taxonomy_id": "skills",
  "version": "v1",
  "skills_count": 68
}

Alias resolution examples
-------------------------
[
  {
    "query": "basic_programming",
    "resolved_to": "programming_fundamentals"
  },
  {
    "query": "programming_basics",
    "resolved_to": "programming_fundamentals"
  },
  {
    "query": "linear_algebra_basics",
    "resolved_to": "linear_algebra"
  },
  {
    "query": "data_structures_and_algorithms",
    "resolved_to": "algorithms_and_data_structures"
  }
]

First 8 taxonomy entries
------------------------
[
  {
    "skill_id": "3d_reconstruction",
    "canonical_label": "3D Scene Reconstruction",
    "aliases": [],
    "category": "applied",
    "level_scale": {
      "min": 0,
      "max": 4
    },
    "status": "active"
  },
  {
    "skill_id": "algorithmic_thinking",
    "canonical_label": "Algorithmic Thinking",
    "aliases": [],
    "category": "applied",
    "level_scale": {
      "min": 0,
      "max": 4
  

## 4. Build the canonical course layer and quality layer

This step keeps the raw layer intact and creates the normalized objects used by the planner.

In [6]:
canonical_courses, quality_snapshots, catalog_issues = build_catalog(REPO_ROOT, TAXONOMY_PATH)
canonical_course = canonical_courses['data_science_technologies']
quality_snapshot = quality_snapshots[canonical_course.quality_ref]

show('Catalog build issues', [issue.model_dump(mode='json') for issue in catalog_issues])
show(
    'Canonical course preview',
    {
        'course_id': canonical_course.course_id,
        'course_name': canonical_course.course_name,
        'quality_ref': canonical_course.quality_ref,
        'domain_scores': [item.model_dump(mode='json') for item in canonical_course.domain_scores],
        'input_skill_refs_count': len(canonical_course.input_skill_refs),
        'output_skill_refs_count': len(canonical_course.output_skill_refs),
        'intra_course_relations_count': len(canonical_course.intra_course_relations),
    },
)
show('First 3 canonical input skill refs', [item.model_dump(mode='json') for item in canonical_course.input_skill_refs], limit=3)
show('Quality snapshot', quality_snapshot)


Catalog build issues
--------------------
[]

Canonical course preview
------------------------
{
  "course_id": "data_science_technologies",
  "course_name": "Data Science Technologies",
  "quality_ref": "course:data_science_technologies",
  "domain_scores": [
    {
      "domain_id": "data_science",
      "domain_label": "Data Science",
      "score": 1.0
    },
    {
      "domain_id": "statistical_learning",
      "domain_label": "Statistical Learning",
      "score": 0.8
    },
    {
      "domain_id": "machine_learning",
      "domain_label": "Machine Learning",
      "score": 0.78
    },
    {
      "domain_id": "decision_support_systems",
      "domain_label": "Decision Support Systems",
      "score": 0.56
    },
    {
      "domain_id": "business_intelligence",
      "domain_label": "Business Intelligence / Data Intelligence",
      "score": 0.42
    },
    {
      "domain_id": "geospatial_analytics",
      "domain_label": "Geospatial Analytics",
      "score": 0.34
    }
  

## 5. Load the semester-based certification program

The program object now contains a track definition, student baseline, semester planning window, course offerings, and inter-course edges.

In [7]:
program = load_program(PROGRAM_PATH)
track = next(track for track in program.tracks if track.track_id == program.track_id)

show(
    'Program summary',
    {
        'program_id': program.program_id,
        'title': program.title,
        'roadmap_mode': program.roadmap_mode,
        'selected_track_id': program.track_id,
        'planning_window': program.planning_window.model_dump(mode='json'),
        'completion_rules': program.completion_rules.model_dump(mode='json'),
    },
)
show(
    'Track summary',
    {
        'track_id': track.track_id,
        'selector_domains': [item.model_dump(mode='json') for item in track.selector_domains],
        'selector_tags': track.selector_tags,
        'target_skill_ids': [item.skill_id for item in track.target_skill_profile],
        'roadmap_policy': track.roadmap_policy.model_dump(mode='json'),
        'manual_includes': track.manual_includes,
    },
)
show(
    'Student baseline',
    {
        'student_id': program.student_profile.student_id,
        'baseline_skill_bank': [item.model_dump(mode='json') for item in program.student_profile.baseline_skill_bank],
        'completed_course_ids': program.student_profile.completed_course_ids,
    },
)
print('Note: this sample MVP assumes Python basics level 3 in the baseline profile because the current 5-course Data Science catalog does not contain a separate Python foundation course.')


Program summary
---------------
{
  "program_id": "data_science_certification_mvp",
  "title": "Data Science Certification MVP",
  "roadmap_mode": "semester",
  "selected_track_id": "data_science",
  "planning_window": {
    "start_term": {
      "course": 1,
      "semester": 1
    },
    "horizon_semesters": 4,
    "max_courses_per_semester": 3
  },
  "completion_rules": {
    "required_slot_ids": [],
    "min_selected_courses": 4,
    "target_coverage_threshold": 0.9
  }
}

Track summary
-------------
{
  "track_id": "data_science",
  "selector_domains": [
    {
      "domain_id": "data_science",
      "weight": 1.0
    },
    {
      "domain_id": "machine_learning",
      "weight": 0.8
    },
    {
      "domain_id": "statistical_learning",
      "weight": 0.7
    },
    {
      "domain_id": "predictive_analytics",
      "weight": 0.5
    },
    {
      "domain_id": "natural_language_processing",
      "weight": 0.4
    },
    {
      "domain_id": "computer_vision",
      "weight"

## 6. Group courses into the chosen track

This is the first real planning step: from the whole catalog, determine which disciplines belong to the selected Data Science track and why.

In [8]:
track_matches = rank_track_courses(program, canonical_courses)
show('Track matches', [match.model_dump(mode='json') for match in track_matches])


Track matches
-------------
[
  {
    "track_id": "data_science",
    "course_id": "analysis_and_processing_of_time_series",
    "affinity_score": 0.3714,
    "role": "core",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "predictive_analytics"
    ],
    "matched_target_skill_ids": [
      "ann_time_series_forecasting",
      "ols_regression",
      "time_series_decomposition"
    ],
    "matched_tags": [
      "data_science",
      "machine_learning",
      "predictive_modeling",
      "statistical_learning"
    ],
    "reason_codes": [
      "domain_alignment",
      "manual_include",
      "tag_similarity",
      "target_skill_overlap"
    ]
  },
  {
    "track_id": "data_science",
    "course_id": "data_science_technologies",
    "affinity_score": 0.3647,
    "role": "foundation",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "statistical_learning"
    ],
    "matched_target_skill_ids": [
      "data_preproces

## 7. Build the semester roadmap

The planner now works semester by semester. It respects:

- max 4 semesters
- max 3 subjects per semester
- offering availability by term
- stage order: foundation -> core -> advanced
- prerequisite accumulation only after a semester is completed

In [9]:
roadmap = build_roadmap(program, canonical_courses, quality_snapshots)
show(
    'Roadmap headline',
    {
        'selected_track_id': roadmap.selected_track_id,
        'selected_course_ids': roadmap.selected_course_ids,
        'achieved_target_coverage': roadmap.achieved_target_coverage,
        'unmet_constraints': roadmap.unmet_constraints,
    },
)
for plan in roadmap.semester_plans:
    show(
        f'Semester {plan.semester_index} @ term {plan.term.course}.{plan.term.semester}',
        {
            'selected_course_ids': plan.selected_course_ids,
            'coverage_after': plan.coverage_after,
            'top_candidates': [
                {
                    'course_id': item.course_id,
                    'score': item.score,
                    'stage': item.stage,
                    'target_skill_gain': item.target_skill_gain,
                    'track_affinity': item.track_affinity,
                    'missing_prerequisites': item.missing_prerequisites,
                }
                for item in plan.candidate_courses
            ],
            'blocked_course_reasons': plan.blocked_course_reasons,
        },
    )


Roadmap headline
----------------
{
  "selected_track_id": "data_science",
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "data_science_technologies",
    "natural_language_analysis_and_processing_nlp",
    "analysis_and_processing_of_time_series",
    "computer_vision_technologies"
  ],
  "achieved_target_coverage": 1.0,
  "unmet_constraints": []
}

Semester 1 @ term 1.1
---------------------
{
  "selected_course_ids": [
    "fundamentals_of_data_science"
  ],
  "coverage_after": 0.1648,
  "top_candidates": [
    {
      "course_id": "fundamentals_of_data_science",
      "score": 0.3547,
      "stage": "foundation",
      "target_skill_gain": 0.1648,
      "track_affinity": 0.18,
      "missing_prerequisites": []
    }
  ],
  "blocked_course_reasons": [
    "analysis_and_processing_of_time_series:not_offered",
    "computer_vision_technologies:not_offered",
    "data_science_technologies:not_offered",
    "natural_language_analysis_and_processing_nlp:not_offered"


## 8. Final student-facing summary

This is the compact artifact the planner returns after it has consumed all selected semesters.

In [10]:
show('Planner summary', roadmap.summary)
target_skill_ids = {item.skill_id for item in track.target_skill_profile}
show(
    'Final target skill bank',
    [item.model_dump(mode='json') for item in roadmap.final_skill_bank if item.skill_id in target_skill_ids],
)
print('Semester-by-semester final path:')
for plan in roadmap.semester_plans:
    print(f"  semester {plan.semester_index} ({plan.term.course}.{plan.term.semester}): {', '.join(plan.selected_course_ids) or 'no courses'}")


Planner summary
---------------
{
  "total_semesters": 4,
  "total_courses": 5,
  "achieved_target_coverage": 1.0,
  "gained_target_skills": [
    "ann_time_series_forecasting",
    "data_preprocessing",
    "data_visualization",
    "digital_image_processing",
    "feature_extraction",
    "image_recognition_and_detection",
    "lemmatization",
    "multicriteria_decision_analysis",
    "ols_regression",
    "similarity_computation",
    "text_vectorization",
    "time_series_decomposition"
  ],
  "unmet_target_skills": []
}

Final target skill bank
-----------------------
[
  {
    "skill_id": "ann_time_series_forecasting",
    "level": 3
  },
  {
    "skill_id": "data_preprocessing",
    "level": 3
  },
  {
    "skill_id": "data_visualization",
    "level": 3
  },
  {
    "skill_id": "digital_image_processing",
    "level": 3
  },
  {
    "skill_id": "feature_extraction",
    "level": 3
  },
  {
    "skill_id": "image_recognition_and_detection",
    "level": 3
  },
  {
    "skill_i

## 9. Infeasibility check: weaken the baseline

This shows the opposite case. If the student starts with weaker Python readiness and the current catalog still has no Python foundation course, the planner should not pretend the track is complete.

In [11]:
weaker_profile = program.student_profile.model_copy(deep=True)
weaker_profile.baseline_skill_bank = [
    item.model_copy(update={'level': 2}) if item.skill_id == 'python_basics' else item
    for item in weaker_profile.baseline_skill_bank
]
weaker_program = program.model_copy(update={'student_profile': weaker_profile}, deep=True)
weaker_roadmap = build_roadmap(weaker_program, canonical_courses, quality_snapshots)

show(
    'Weaker-baseline roadmap result',
    {
        'selected_course_ids': weaker_roadmap.selected_course_ids,
        'achieved_target_coverage': weaker_roadmap.achieved_target_coverage,
        'unmet_constraints': weaker_roadmap.unmet_constraints,
    },
)
for plan in weaker_roadmap.semester_plans:
    print(f"semester {plan.semester_index} blocked reasons:")
    pprint(plan.blocked_course_reasons)



Weaker-baseline roadmap result
------------------------------
{
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "analysis_and_processing_of_time_series",
    "natural_language_analysis_and_processing_nlp",
    "computer_vision_technologies"
  ],
  "achieved_target_coverage": 0.7143,
  "unmet_constraints": [
    "external_prereq_gap:data_science_technologies:python_basics:3",
    "semester_2:no_eligible_courses",
    "target_coverage_below_policy"
  ]
}
semester 1 blocked reasons:
['analysis_and_processing_of_time_series:not_offered',
 'computer_vision_technologies:not_offered',
 'data_science_technologies:not_offered',
 'natural_language_analysis_and_processing_nlp:not_offered']
semester 2 blocked reasons:
['analysis_and_processing_of_time_series:not_offered',
 'computer_vision_technologies:not_offered',
 'data_science_technologies:missing_prerequisites',
 'natural_language_analysis_and_processing_nlp:not_offered']
semester 3 blocked reasons:
['computer_vision_techn

## 10. CLI walkthrough

The same pipeline is also exposed through the public CLI. This is useful when you want to validate the repo or rebuild the roadmap without importing Python modules manually.

In [12]:
from subprocess import run

cli_commands = [
    ['python3', '-m', 'roadmaps_mvp.cli', 'validate-courses'],
    ['python3', '-m', 'roadmaps_mvp.cli', 'rank-track-courses'],
    ['python3', '-m', 'roadmaps_mvp.cli', 'build-roadmap'],
]

for command in cli_commands:
    print(f"\n$ PYTHONPATH=src {' '.join(command)}")
    completed = run(
        command,
        cwd=str(REPO_ROOT),
        env={'PYTHONPATH': str(SRC_DIR), **__import__('os').environ},
        capture_output=True,
        text=True,
        check=True,
    )
    print(completed.stdout[:4000])


$ PYTHONPATH=src python3 -m roadmaps_mvp.cli validate-courses
{
  "validated_courses": 5,
  "error_count": 0,
  "warning_count": 0,
  "errors": [],
  "warnings": []
}


$ PYTHONPATH=src python3 -m roadmaps_mvp.cli rank-track-courses
[
  {
    "track_id": "data_science",
    "course_id": "analysis_and_processing_of_time_series",
    "affinity_score": 0.3714,
    "role": "core",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "predictive_analytics"
    ],
    "matched_target_skill_ids": [
      "ann_time_series_forecasting",
      "ols_regression",
      "time_series_decomposition"
    ],
    "matched_tags": [
      "data_science",
      "machine_learning",
      "predictive_modeling",
      "statistical_learning"
    ],
    "reason_codes": [
      "domain_alignment",
      "manual_include",
      "tag_similarity",
      "target_skill_overlap"
    ]
  },
  {
    "track_id": "data_science",
    "course_id": "data_science_technologies",
    "affinity_scor

## 11. Generated files

The CLI writes the final roadmap report and keeps the raw, canonical, taxonomy, and quality layers separate.

In [13]:
show('Latest roadmap report', load_json(REPORT_PATH))
print('Useful files:')
print('  raw layer:            ', REPO_ROOT)
print('  taxonomy layer:       ', TAXONOMY_PATH)
print('  program layer:        ', PROGRAM_PATH)
print('  latest report:        ', REPORT_PATH)
print('  this notebook:        ', NOTEBOOK_PATH)


Latest roadmap report
---------------------
{
  "program_id": "data_science_certification_mvp",
  "selected_track_id": "data_science",
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "data_science_technologies",
    "natural_language_analysis_and_processing_nlp",
    "analysis_and_processing_of_time_series",
    "computer_vision_technologies"
  ],
  "slot_results": [],
  "semester_plans": [
    {
      "semester_index": 1,
      "term": {
        "course": 1,
        "semester": 1
      },
      "selected_course_ids": [
        "fundamentals_of_data_science"
      ],
      "selected_courses": [
        {
          "course_id": "fundamentals_of_data_science",
          "course_name": "Fundamentals of Data Science",
          "score": 0.3547,
          "track_affinity": 0.18,
          "prerequisite_fit": 1.0,
          "readiness_score": 1.0,
          "target_skill_gain": 0.1648,
          "unlock_score": 0.0,
          "domain_alignment": 0.18,
          "phase_fit

## 12. Skill-based student profile and elective recommendations

The new layer keeps track context, but recommendations now also depend on the student's mandatory curriculum, current progress, and manually added skills normalized through the taxonomy.

In [14]:
from roadmaps_mvp.models import AcademicTerm
from roadmaps_mvp.student_recommendations import (
    build_runtime_skill_resolver,
    build_student_skill_profile,
    recommend_electives,
    run_manual_skill_input_session,
)

SPECIALIZATION_REQUIRED_DIR = REPO_ROOT / 'student_db' / 'base_subjects' / '121_ipi'
SPECIALIZATION_REQUIRED_FIRST_COURSE_DIR = SPECIALIZATION_REQUIRED_DIR / '1st_course'

print('Specialization curriculum root:', SPECIALIZATION_REQUIRED_DIR)
print('First-course example dir:', SPECIALIZATION_REQUIRED_FIRST_COURSE_DIR)

Specialization curriculum root: /Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi
First-course example dir: /Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi/1st_course


In [15]:
student_profile = build_student_skill_profile(
    specialization_id='data_science',
    current_course=1,
    current_semester=1,
    required_subject_dir=SPECIALIZATION_REQUIRED_DIR,
    elective_raw_dir=REPO_ROOT,
    manual_skill_inputs=['programming basics', 'text vectorization', 'mystery skill'],
)

show(
    'Student skill profile summary',
    {
        'specialization_id': student_profile.specialization_id,
        'current_term': f"{student_profile.current_course}.{student_profile.current_semester}",
        'required_courses_count': len(student_profile.required_courses),
        'base_skills_count': len(student_profile.automatically_extracted_base_skills),
        'current_curriculum_skills_count': len(student_profile.current_curriculum_skills),
        'planned_curriculum_skills_count': len(student_profile.planned_curriculum_skills),
        'user_skills_count': len(student_profile.user_skills),
        'unrecognized_user_skill_inputs': student_profile.unrecognized_user_skill_inputs,
    },
)
show(
    'First 8 required courses with status',
    [item.model_dump(mode='json') for item in student_profile.required_courses],
    limit=8,
)
show(
    'Normalized user skills',
    [item.model_dump(mode='json') for item in student_profile.user_skills],
)


Student skill profile summary
-----------------------------
{
  "specialization_id": "data_science",
  "current_term": "1.1",
  "required_courses_count": 10,
  "base_skills_count": 68,
  "current_curriculum_skills_count": 33,
  "planned_curriculum_skills_count": 36,
  "user_skills_count": 2,
  "unrecognized_user_skill_inputs": [
    "mystery skill"
  ]
}

First 8 required courses with status
------------------------------------
[
  {
    "course_id": "algorithms_and_data_structures_part_1_basics_of_algorithmization",
    "course_name": "Algorithms and Data Structures. Part 1. Basics of Algorithmization",
    "course": 1,
    "semester": 1,
    "status": "in_progress",
    "raw_source_file": "/Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi/1st_course/algorithms_and_data_structures_part_1_basics_of_algorithmization.json"
  },
  {
    "course_id": "computer_discrete_mathematics",
    "course_name": "Computer Discrete Mathematics",
    "course": 1,
    "sem

In [16]:
runtime_resolver = build_runtime_skill_resolver(SPECIALIZATION_REQUIRED_DIR, REPO_ROOT)
manual_session = run_manual_skill_input_session(
    auto_skills=student_profile.automatically_extracted_base_skills,
    manual_skill_rounds=[['programming basics'], ['linear algebra basics'], ['None', 'ignored after stop']],
    resolver=runtime_resolver,
)

show('Manual input session trace', manual_session)


Manual input session trace
--------------------------
{
  "auto_skills_shown": [
    "advanced_sorting_algorithms",
    "agile_process_management",
    "algorithm_complexity_analysis",
    "algorithm_design",
    "analytic_geometry_modeling",
    "boolean_function_minimization",
    "c_programming",
    "cisco_ios_administration",
    "combinatorial_algorithms",
    "complex_number_manipulation",
    "computational_complexity_theory",
    "csharp_programming",
    "data_structure_implementation",
    "data_structures_fundamentals",
    "definite_integration",
    "dynamic_memory_management",
    "dynamic_programming",
    "dynamic_routing_configuration",
    "event_driven_programming_basics",
    "exception_handling",
    "file_io_operations",
    "finite_state_machine_design",
    "flowchart_diagramming",
    "flowcharting",
    "formal_language_theory",
    "function_analysis_and_graphing",
    "function_differentiation",
    "geometric_integration_applications",
    "graph_algorith

In [17]:
skill_based_recommendations = recommend_electives(
    profile=student_profile,
    program=program,
    courses=canonical_courses,
    quality_snapshots=quality_snapshots,
    required_subject_dir=SPECIALIZATION_REQUIRED_DIR,
    elective_raw_dir=REPO_ROOT,
    term_capacities={(1, 1): 0, (1, 2): 1, (2, 1): 2, (2, 2): 1},
    electives_start_term=AcademicTerm(course=1, semester=2),
)

show(
    'Skill-based recommendation summary',
    {
        'terms': [
            {
                'term': item.term.model_dump(mode='json'),
                'max_electives': item.max_electives,
                'recommended_course_ids': item.recommended_course_ids,
            }
            for item in skill_based_recommendations.term_recommendations
        ],
    },
)
show(
    'First term with active recommendations',
    next(
        item.model_dump(mode='json')
        for item in skill_based_recommendations.term_recommendations
        if item.candidate_courses
    ),
)


Skill-based recommendation summary
----------------------------------
{
  "terms": [
    {
      "term": {
        "course": 1,
        "semester": 1
      },
      "max_electives": 0,
      "recommended_course_ids": []
    },
    {
      "term": {
        "course": 1,
        "semester": 2
      },
      "max_electives": 1,
      "recommended_course_ids": [
        "data_science_technologies"
      ]
    },
    {
      "term": {
        "course": 2,
        "semester": 1
      },
      "max_electives": 2,
      "recommended_course_ids": [
        "analysis_and_processing_of_time_series",
        "natural_language_analysis_and_processing_nlp"
      ]
    },
    {
      "term": {
        "course": 2,
        "semester": 2
      },
      "max_electives": 1,
      "recommended_course_ids": [
        "computer_vision_technologies"
      ]
    }
  ]
}

First term with active recommendations
--------------------------------------
{
  "term": {
    "course": 1,
    "semester": 2
  },
  "max_

## 13. CLI checks for the skill-based layer

These commands exercise the new public CLI surface for student-profile construction and elective recommendation.

In [18]:
skill_cli_commands = [
    [
        'python3', '-m', 'roadmaps_mvp.cli', 'build-student-profile',
        '--specialization-id', 'data_science',
        '--current-course', '1',
        '--current-semester', '1',
        '--required-subject-dir', str(SPECIALIZATION_REQUIRED_DIR),
        '--manual-skill', 'programming basics',
    ],
    [
        'python3', '-m', 'roadmaps_mvp.cli', 'recommend-electives',
        '--specialization-id', 'data_science',
        '--current-course', '1',
        '--current-semester', '1',
        '--required-subject-dir', str(SPECIALIZATION_REQUIRED_DIR),
        '--electives-start-course', '1',
        '--electives-start-semester', '2',
        '--term-capacity', '1:1=0',
        '--term-capacity', '1:2=1',
    ],
]

for command in skill_cli_commands:
    print(f"\n$ PYTHONPATH=src {' '.join(command)}")
    completed = run(
        command,
        cwd=str(REPO_ROOT),
        env={'PYTHONPATH': str(SRC_DIR), **__import__('os').environ},
        capture_output=True,
        text=True,
        check=True,
    )
    print(completed.stdout[:4000])


$ PYTHONPATH=src python3 -m roadmaps_mvp.cli build-student-profile --specialization-id data_science --current-course 1 --current-semester 1 --required-subject-dir /Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi --manual-skill programming basics
{
  "output": "reports/student_skill_profile.json",
  "specialization_id": "data_science",
  "current_course": 1,
  "current_semester": 1,
  "required_courses": [
    {
      "course_id": "algorithms_and_data_structures_part_1_basics_of_algorithmization",
      "course_name": "Algorithms and Data Structures. Part 1. Basics of Algorithmization",
      "course": 1,
      "semester": 1,
      "status": "in_progress",
      "raw_source_file": "/Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi/1st_course/algorithms_and_data_structures_part_1_basics_of_algorithmization.json"
    },
    {
      "course_id": "computer_discrete_mathematics",
      "course_name": "Computer Discrete Mathematics

## 12. Skill-based layer: specialization curriculum and student profile

This section exercises the new skill-based flow on the specialization curriculum stored under `student_db/base_subjects/121_ipi/1st_course`. It keeps the existing roadmap walkthrough intact and adds a focused check for the student-profile layer.


In [19]:
from roadmaps_mvp.models import AcademicTerm
from roadmaps_mvp.student_recommendations import (
    build_student_skill_profile,
    recommend_electives,
    run_manual_skill_input_session,
)

SPECIALIZATION_REQUIRED_DIR = REPO_ROOT / 'student_db' / 'base_subjects' / '121_ipi' / '1st_course'
SPECIALIZATION_ID = 'data_science'
CURRENT_TERM = AcademicTerm(course=1, semester=2)

print('Specialization curriculum dir:', SPECIALIZATION_REQUIRED_DIR)
print('Exists:', SPECIALIZATION_REQUIRED_DIR.exists())
print('Current student term:', CURRENT_TERM.model_dump(mode='json'))


Specialization curriculum dir: /Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi/1st_course
Exists: True
Current student term: {'course': 1, 'semester': 2}


## 13. Build the student profile from required specialization subjects

This should collect required courses, classify them into `completed / in_progress / planned`, extract curriculum skills, and expose a combined student-facing profile object.


In [20]:
profile_from_curriculum = build_student_skill_profile(
    specialization_id=SPECIALIZATION_ID,
    current_course=CURRENT_TERM.course,
    current_semester=CURRENT_TERM.semester,
    required_subject_dir=SPECIALIZATION_REQUIRED_DIR,
    elective_raw_dir=REPO_ROOT,
)

required_status_counts = {}
for item in profile_from_curriculum.required_courses:
    required_status_counts[item.status] = required_status_counts.get(item.status, 0) + 1

show(
    'Student profile from specialization curriculum',
    {
        'specialization_id': profile_from_curriculum.specialization_id,
        'current_course': profile_from_curriculum.current_course,
        'current_semester': profile_from_curriculum.current_semester,
        'required_courses_total': len(profile_from_curriculum.required_courses),
        'required_status_counts': required_status_counts,
        'auto_base_skills_total': len(profile_from_curriculum.automatically_extracted_base_skills),
        'current_curriculum_skills_total': len(profile_from_curriculum.current_curriculum_skills),
        'planned_curriculum_skills_total': len(profile_from_curriculum.planned_curriculum_skills),
        'combined_skill_profile_total': len(profile_from_curriculum.combined_skill_profile),
    },
)
show(
    'First 5 required specialization subjects',
    [item.model_dump(mode='json') for item in profile_from_curriculum.required_courses[:5]],
)
show(
    'First 10 automatically extracted skills',
    [item.model_dump(mode='json') for item in profile_from_curriculum.automatically_extracted_base_skills[:10]],
)



Student profile from specialization curriculum
----------------------------------------------
{
  "specialization_id": "data_science",
  "current_course": 1,
  "current_semester": 2,
  "required_courses_total": 10,
  "required_status_counts": {
    "completed": 5,
    "in_progress": 5
  },
  "auto_base_skills_total": 68,
  "current_curriculum_skills_total": 68,
  "planned_curriculum_skills_total": 0,
  "combined_skill_profile_total": 68
}

First 5 required specialization subjects
----------------------------------------
[
  {
    "course_id": "algorithms_and_data_structures_part_1_basics_of_algorithmization",
    "course_name": "Algorithms and Data Structures. Part 1. Basics of Algorithmization",
    "course": 1,
    "semester": 1,
    "status": "completed",
    "raw_source_file": "/Users/pelmeshek1706/Desktop/projects/roadmaps/student_db/base_subjects/121_ipi/1st_course/algorithms_and_data_structures_part_1_basics_of_algorithmization.json"
  },
  {
    "course_id": "computer_discrete

## 14. Manual skills and their normalization

The new layer should support a loop where the system shows auto-detected skills, accepts extra manual inputs, normalizes them against the taxonomy, reports recognized additions, and stops when the student enters `None`.


In [21]:
manual_skill_rounds = [
    ['basic_programming', 'Python Basics'],
    ['linear algebra basics', 'totally unknown skill', 'None'],
]

manual_skill_session = run_manual_skill_input_session(
    auto_skills=profile_from_curriculum.automatically_extracted_base_skills,
    manual_skill_rounds=manual_skill_rounds,
    resolver=SkillResolver(load_taxonomy(TAXONOMY_PATH)),
)

profile_with_manual_skills = build_student_skill_profile(
    specialization_id=SPECIALIZATION_ID,
    current_course=CURRENT_TERM.course,
    current_semester=CURRENT_TERM.semester,
    required_subject_dir=SPECIALIZATION_REQUIRED_DIR,
    elective_raw_dir=REPO_ROOT,
    manual_skill_rounds=manual_skill_rounds,
)

show(
    'Manual skill session readout',
    {
        'auto_skills_shown_count': len(manual_skill_session.auto_skills_shown),
        'rounds': [round_item.model_dump(mode='json') for round_item in manual_skill_session.rounds],
        'collected_user_skills': [item.model_dump(mode='json') for item in manual_skill_session.collected_user_skills],
        'unrecognized_inputs': manual_skill_session.unrecognized_inputs,
    },
)
show(
    'Merged user skill profile',
    {
        'user_skills': [item.model_dump(mode='json') for item in profile_with_manual_skills.user_skills],
        'unrecognized_user_skill_inputs': profile_with_manual_skills.unrecognized_user_skill_inputs,
    },
)



Manual skill session readout
----------------------------
{
  "auto_skills_shown_count": 68,
  "rounds": [
    {
      "shown_auto_skills": [
        "advanced_sorting_algorithms",
        "agile_process_management",
        "algorithm_complexity_analysis",
        "algorithm_design",
        "analytic_geometry_modeling",
        "boolean_function_minimization",
        "c_programming",
        "cisco_ios_administration",
        "combinatorial_algorithms",
        "complex_number_manipulation",
        "computational_complexity_theory",
        "csharp_programming",
        "data_structure_implementation",
        "data_structures_fundamentals",
        "definite_integration",
        "dynamic_memory_management",
        "dynamic_programming",
        "dynamic_routing_configuration",
        "event_driven_programming_basics",
        "exception_handling",
        "file_io_operations",
        "finite_state_machine_design",
        "flowchart_diagramming",
        "flowcharting",
    

## 15. Recommend electives with external term capacity and delayed elective start

Here we pass two external planning controls:

- `term_capacities`: max electives per term
- `electives_start_term`: the first term where electives are actually allowed

This lets the notebook demonstrate both scheduling capacity and the rule that electives may start later than the current term.


In [22]:
term_capacities = {
    (1, 2): 1,
    (2, 1): 2,
    (2, 2): 1,
    (3, 1): 1,
}
electives_start_term = AcademicTerm(course=2, semester=1)

elective_recommendations = recommend_electives(
    profile=profile_with_manual_skills,
    program=program,
    courses=canonical_courses,
    quality_snapshots=quality_snapshots,
    required_subject_dir=SPECIALIZATION_REQUIRED_DIR,
    elective_raw_dir=REPO_ROOT,
    term_capacities=term_capacities,
    electives_start_term=electives_start_term,
)

show(
    'Elective recommendation headline',
    {
        'specialization_id': elective_recommendations.specialization_id,
        'current_term': elective_recommendations.current_term.model_dump(mode='json'),
        'electives_start_term': electives_start_term.model_dump(mode='json'),
        'term_capacities': {f'{course}.{semester}': cap for (course, semester), cap in term_capacities.items()},
    },
)

for plan in elective_recommendations.term_recommendations:
    show(
        f'Elective term {plan.term.course}.{plan.term.semester}',
        {
            'max_electives': plan.max_electives,
            'recommended_course_ids': plan.recommended_course_ids,
            'top_candidates': [
                {
                    'course_id': item.course_id,
                    'score': item.score,
                    'reason_codes': item.reason_codes,
                    'developed_skill_ids': item.developed_skill_ids,
                    'overlapping_skill_ids': item.overlapping_skill_ids,
                    'missing_prerequisites': item.missing_prerequisites,
                }
                for item in plan.candidate_courses[:3]
            ],
            'blocked_course_reasons': plan.blocked_course_reasons,
        },
    )



Elective recommendation headline
--------------------------------
{
  "specialization_id": "data_science",
  "current_term": {
    "course": 1,
    "semester": 2
  },
  "electives_start_term": {
    "course": 2,
    "semester": 1
  },
  "term_capacities": {
    "1.2": 1,
    "2.1": 2,
    "2.2": 1,
    "3.1": 1
  }
}

Elective term 1.2
-----------------
{
  "max_electives": 1,
  "recommended_course_ids": [],
  "top_candidates": [],
  "blocked_course_reasons": [
    "analysis_and_processing_of_time_series:electives_locked_until:2:1",
    "computer_vision_technologies:electives_locked_until:2:1",
    "data_science_technologies:electives_locked_until:2:1",
    "fundamentals_of_data_science:electives_locked_until:2:1",
    "natural_language_analysis_and_processing_nlp:electives_locked_until:2:1"
  ]
}

Elective term 2.1
-----------------
{
  "max_electives": 2,
  "recommended_course_ids": [
    "data_science_technologies",
    "analysis_and_processing_of_time_series"
  ],
  "top_candidate

## 16. Sanity check / compact readout

A short final readout helps verify that the new layer behaves coherently before moving to deeper inspection.


In [23]:
sanity = {
    'required_courses_total': len(profile_with_manual_skills.required_courses),
    'recognized_manual_skill_ids': [item.skill_id for item in profile_with_manual_skills.user_skills],
    'unrecognized_manual_inputs': profile_with_manual_skills.unrecognized_user_skill_inputs,
    'term_summary': [
        {
            'term': f'{plan.term.course}.{plan.term.semester}',
            'max_electives': plan.max_electives,
            'recommended_count': len(plan.recommended_course_ids),
            'recommended_course_ids': plan.recommended_course_ids,
        }
        for plan in elective_recommendations.term_recommendations
    ],
}
show('Skill-based layer sanity check', sanity)

assert sanity['required_courses_total'] == 10
assert 'programming_fundamentals' in sanity['recognized_manual_skill_ids']
assert 'python_basics' in sanity['recognized_manual_skill_ids']
assert elective_recommendations.term_recommendations[0].recommended_course_ids == []
assert all(
    len(plan.recommended_course_ids) <= plan.max_electives
    for plan in elective_recommendations.term_recommendations
)
assert all(
    (plan.term.course, plan.term.semester) >= (electives_start_term.course, electives_start_term.semester)
    or not plan.recommended_course_ids
    for plan in elective_recommendations.term_recommendations
)
print('Sanity checks passed.')



Skill-based layer sanity check
------------------------------
{
  "required_courses_total": 10,
  "recognized_manual_skill_ids": [
    "linear_algebra",
    "programming_fundamentals",
    "python_basics"
  ],
  "unrecognized_manual_inputs": [
    "totally unknown skill"
  ],
  "term_summary": [
    {
      "term": "1.2",
      "max_electives": 1,
      "recommended_count": 0,
      "recommended_course_ids": []
    },
    {
      "term": "2.1",
      "max_electives": 2,
      "recommended_count": 2,
      "recommended_course_ids": [
        "data_science_technologies",
        "analysis_and_processing_of_time_series"
      ]
    },
    {
      "term": "2.2",
      "max_electives": 1,
      "recommended_count": 1,
      "recommended_course_ids": [
        "natural_language_analysis_and_processing_nlp"
      ]
    },
    {
      "term": "3.1",
      "max_electives": 1,
      "recommended_count": 0,
      "recommended_course_ids": []
    }
  ]
}
Sanity checks passed.
